## YOLOv10 + SAM

In [1]:
import os
os.getcwd()
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")

In [2]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from Solar_Rooftop_Detection.accuracy import compute_metrics
from Solar_Rooftop_Detection.generate_isolated_masks import generate_isolated_mask

In [3]:
# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/yolov10/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/sam_check_point/Final_Models/FineTune_model_0_epoch_28_02_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv10_SAM.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

for img_name in os.listdir(data_folder):
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    mask_path = os.path.join(mask_folder, img_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)


image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_338674_1201408.jpg: 1024x1024 2 panels, 5.0ms
Speed: 3.9ms preprocess, 5.0ms inference, 14.5ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_318899_1218618.jpg: 1024x1024 (no detections), 6.1ms
Speed: 3.4ms preprocess, 6.1ms inference, 2.5ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_332122_1180408.jpg: 1024x1024 2 panels, 4.8ms
Speed: 3.6ms preprocess, 4.8ms inference, 3.6ms postprocess per image at shape (1, 3, 1024,

In [4]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv10_SAM.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.878288
pixel_dice                 0.922952
pixel_accuracy             0.989098
pixel_precision            0.893993
pixel_recall               0.980446
region_iou                 0.878288
region_dice                0.922952
region_precision           0.893993
region_recall              0.980446
region_success_accuracy    0.974359
dtype: float64

## YOLOv11 + SAM

In [5]:
import os
os.getcwd()
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")

import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from Solar_Rooftop_Detection.accuracy import compute_metrics
from Solar_Rooftop_Detection.generate_isolated_masks import generate_isolated_mask

In [6]:
# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/yolov11/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/sam_check_point/Final_Models/FineTune_model_0_epoch_28_02_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv11_SAM.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)


results_data = []

for img_name in os.listdir(data_folder):
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    mask_path = os.path.join(mask_folder, img_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)


image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_338674_1201408.jpg: 1024x1024 2 panels, 6.2ms
Speed: 1.8ms preprocess, 6.2ms inference, 28.3ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_318899_1218618.jpg: 1024x1024 (no detections), 5.9ms
Speed: 1.3ms preprocess, 5.9ms inference, 3.7ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_332122_1180408.jpg: 1024x1024 1 panel, 6.2ms
Speed: 3.7ms preprocess, 6.2ms inference, 4.2ms postprocess per image at shape (1, 3, 1024, 

In [7]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv11_SAM.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.881936
pixel_dice                 0.928110
pixel_accuracy             0.989532
pixel_precision            0.900556
pixel_recall               0.977226
region_iou                 0.881936
region_dice                0.928110
region_precision           0.900556
region_recall              0.977226
region_success_accuracy    0.979701
dtype: float64

# YOLOv12 + SAM

In [1]:
import os
os.getcwd()
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")

import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from ultralytics import YOLO
import os 
from Solar_Rooftop_Detection.accuracy import compute_metrics
from Solar_Rooftop_Detection.generate_isolated_masks import generate_isolated_mask

# Load YOLO model 
yolo_model = YOLO("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/yolov12_panel/weights/best.pt")

# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/sam_check_point/Final_Models/FineTune_model_0_epoch_28_02_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv12_SAM.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)


results_data = []

for img_name in os.listdir(data_folder):
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    yolo_results = yolo_model.predict(img_path, conf=0.5, device="cuda")
    boxes = yolo_results[0].boxes.xyxy.cpu().numpy()
    
    mask_path = os.path.join(mask_folder, img_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_338674_1201408.jpg: 1024x1024 2 panels, 4.1ms
Speed: 1.9ms preprocess, 4.1ms inference, 53.8ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_318899_1218618.jpg: 1024x1024 (no detections), 4.5ms
Speed: 1.7ms preprocess, 4.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1024, 1024)

image 1/1 /home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images/PV03_332122_1180408.jpg: 1024x1024 2 panels, 4.0ms

In [2]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/YOLOv12_SAM.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.880547
pixel_dice                 0.926848
pixel_accuracy             0.989002
pixel_precision            0.898385
pixel_recall               0.976996
region_iou                 0.880547
region_dice                0.926848
region_precision           0.898385
region_recall              0.976996
region_success_accuracy    0.979636
dtype: float64

# DETR + SAM

In [8]:
import os
os.getcwd()
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")

import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from Solar_Rooftop_Detection.accuracy import compute_metrics
from Solar_Rooftop_Detection.generate_isolated_masks import generate_isolated_mask
from rfdetr import RFDETRBase
from PIL import Image

E0000 00:00:1746261091.159385  204640 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746261091.161699  204640 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746261091.168009  204640 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746261091.168018  204640 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746261091.168020  204640 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746261091.168020  204640 computation_placer.cc:177] computation placer already registered. Please check linka

In [9]:
# Load SAM model
# Load SAM model 
sam = sam_model_registry["vit_b"]("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/sam_check_point/Final_Models/FineTune_model_0_epoch_28_02_05_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/images"
mask_folder = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/SOLAR_PANEL_DATASET/test/masks"
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/DETR_SAM.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

checkpoint_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/detr_panel/checkpoint_best_total.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_wrapper = RFDETRBase()

underlying_model = model_wrapper.model
underlying_model.reinitialize_detection_head(num_classes=2)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pytorch_model = underlying_model.model
pytorch_model.load_state_dict(checkpoint['model'])

pytorch_model.to(device)
pytorch_model.eval()

for img_name in os.listdir(data_folder):
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    detections = model_wrapper.predict(image, threshold=0.5)
    boxes = detections.xyxy
    
    mask_path = os.path.join(mask_folder, img_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

Loading pretrain weights


In [10]:
metrics = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_SOLAR_PANEL_SEGMENTATION/results/sam/DETR_SAM.csv")
output = metrics.mean(axis=0)
output

pixel_iou                  0.843871
pixel_dice                 0.888461
pixel_accuracy             0.975882
pixel_precision            0.858029
pixel_recall               0.982312
region_iou                 0.843871
region_dice                0.888461
region_precision           0.858029
region_recall              0.982312
region_success_accuracy    0.936893
dtype: float64